# Data analysis for ingest pipeline

Validate that TasteMender's imported audio-feature data is complete, internally consistent, and faithful to the original AcousticBrainz source data.

This work should identify any issues introduced during ingestion, transformation, normalization, or storage, and establish whether each feature provides enough useful variation to support recommendation ranking.

Links:
- https://acousticbrainz.org/datasets/accuracy

In [ ]:
import os, uuid, sys, random
import django
import numpy as np
import numpy.typing as npt
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from IPython.display import Markdown, display

# Enable access to Django project inside current notebook
PROJECT_PATH = os.path.abspath(os.path.join("..", ".."))
sys.path.append(PROJECT_PATH)
os.environ["DJANGO_SETTINGS_MODULE"] = "music_recommendation.settings"
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"
django.setup()

# Ensure the latest version of analysis_helpers is imported
%load_ext autoreload
%autoreload 1
%aimport ingest.analysis.analysis_helpers

from django.conf import settings
from recommend_api.models import GenreDortmund, GenreRosamerica, Track, Artist, Album
import ingest.analysis.analysis_helpers as analysis_helpers
from ingest.analysis.analysis_helpers import export_stats
from ingest.core.cli_helpers import fg, bg, style

### Genre distribution by classifier

In [ ]:
feature_store = np.load(settings.BASE_DIR / settings.FEATURE_MATRIX_FILENAME, allow_pickle=True, mmap_mode="r")

genre_display_names = {
    "folkcountry": "Folk / Country",
    "funksoulrnb": "Funk / Soul / R&B",
    "raphiphop": "Rap / Hip-Hop",
    "cla": "Classical",
    "dan": "Dance",
    "hip": "Hip-Hop",
    "jaz": "Jazz",
    "rhy": "Rhythm & Blues",
    "roc": "Rock",
    "spe": "Speech",
}
genre_classifiers = (
    ("Dortmund", "genre_dortmund", GenreDortmund),
    ("Rosamerica", "genre_rosamerica", GenreRosamerica),
)

fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharex=True, layout="constrained")
for axis, (classifier, feature_key, genre_model) in zip(axes, genre_classifiers):
    shares = pd.Series(feature_store[feature_key]).value_counts(normalize=True) * 100
    labels_by_code = dict(genre_model.objects.values_list("code", "label"))
    labels = [labels_by_code[code] for code in shares.index]
    names = [genre_display_names.get(label, label.title()) for label in labels]
    bars = axis.barh(names, shares)
    axis.invert_yaxis()
    share_labels = [f"{share:.1f}%" if share >= 0.1 else "<0.1%" for share in shares]
    axis.bar_label(bars, labels=share_labels, padding=4)
    axis.set(title=classifier, xlabel="Share of tracks (%)", xlim=(0, 100))

plt.show()

The Dortmund classification designates a whopping 77.8% as being electronic music while Rosamerica splits up genres more evenly. The official AcousticBrainz [accuracy metrics](https://acousticbrainz.org/datasets/accuracy) put Dortmund at 60.25% and Rosamerica at 87.56%.

We can safely conclude that the Rosamerica classification is the best choice for determining musical genre in TasteMender due to its higher accuracy.

### Feature Distribution - Normalized vs Raw
Audio features go through 3 different representations before they can be used by the recommender logic. They are:
- **raw** - the original audio features that are extracted from AcousticBrainz JSON dataset
- **standardized** - each feature is transformed to have mean 0 and standard deviation 1 using `StandardScaler`
- **normalized** - the complete feature vector for each track is divided by its Euclidean norm for cosine similarity

It's currently unclear if and how changing representations affects the recommendation logic. Since this is the core of the project, it worthwhile to make a deep dive into this aspect as well as the logic in general. We'll begin by comparing the representations for all of the audio features.

Questions that should be answered:
- How are features distributed in general?
- Does standardization and normalization change the shape of the distribution?

Note: The DB ingestion pipeline skips submissions that are missing metadata. For example Pharrell Williams' "Happy" has 326 submissions but only 320 have valid metadata. Keep this discrepancy in mind as the "submissions" count in the `Track` table can be lower than number of records in the original dataset.

In [ ]:
# Hardcode lists of MBIDs for analysis and get their aggregate stats
danceable_mbids = [
    "d2bab0db-b63b-45b6-8f6c-5a566620c522",  # Mambo No. 5 (A Little Bit of...) - Lou Bega
    "c7a0fed2-988b-43d2-b7ff-4055fcad868f",  # Conga - Miami Sound Machine
    "0d8df525-cc5a-4ee4-a303-e4df9ca0a685",  # Freestyler - Bomfunk MC's
    "a502f559-f70c-4e0d-a5e5-e0cc319d0bd7",  # Dancing Queen - ABBA
    "60fa767a-d85d-4991-82bc-4294e0b11ae7",  # One More Time - Daft Punk
    "24cbf6b7-bcc9-42d3-aa61-76e637bfde8c",  # Crimson and Clover - Tommy James & The Shondells
    "8121ce09-135d-477d-aafb-0404f1f15b78",  # Blue in Green - Miles Davis
    "141b174e-ee58-4447-990a-6806180b455b",  # The Sound of Silence - Simon & Garfunkel
    "d7bb88a7-d9ca-4481-9e79-5ae17c0daefb",  # Adagio for Strings - Samuel Barber
    "acc8f927-65e9-4662-8665-8127efe76e6a",  # Foil - Autechre
]
aggressive_mbids = [
    "62c2e20a-559e-422f-a44c-9afa7882f0c4",  # Enter Sandman - Metallica
    "c7c25302-69ab-4a9d-b967-23cbc6793854",  # Hells Bells - AC/DC
    "06748144-5e3a-4ce3-ae4f-3d15768089af",  # Hearts on Fire - HammerFall
    "03cf6e1b-d986-402f-b67e-9fbe2ebc906d",  # Burning Times - Iced Earth
    "5053c294-90fe-47e9-9e21-1a4c4538c20f",  # Bad Medicine - Bon Jovi
    "37295d29-229a-4609-bb53-0a72e5e0a467",  # 2 Minutes to Midnight - Iron Maiden
    "71209300-34e5-491d-81ad-76237d167604",  # Ace of Spades - Motörhead
    "9190aac9-cbb1-4f26-8b4c-169e08d7897a",  # Bed of Roses - Bon Jovi
    "b1a9c0e9-d987-4042-ae91-78d6a3267d69",  # Bohemian Rhapsody - Queen
    "2dacc772-bff6-4347-a586-8bff3a7d7c79",  # Nothing Else Matters - Metallica
]
# For each title/artist pair, use the recording with the most stored submissions.
happy_mbids = [
    "f367ad6e-efd5-4496-aba5-21a4edcd82c6",  # Happy - Pharrell Williams
    "d4009c09-a339-4b89-bc45-d91ab649acb6",  # Don't Stop Me Now - Queen
    "2138bf67-fe08-46d3-863d-560e1b8c3043",  # Walking on Sunshine - Katrina and the Waves
    "3ad31947-ae54-4409-aeaa-8dbd16c3825f",  # Good Vibrations - The Beach Boys
    "440f60e8-0b25-4ec4-abb1-c6beec624ab0",  # Here Comes the Sun - The Beatles
    "6e33056b-2a34-4ed0-b49c-f7a8ae2c5bcc",  # Mr. Blue Sky - Electric Light Orchestra
    "2d6daa5a-19d9-4254-b149-26f162c4634f",  # Lovely Day - Bill Withers
    "25d8de5e-3662-4ffd-8dea-511a696ac3e7",  # Hurt - Johnny Cash
    "49eb0679-0a3d-4be5-9adf-973cd429b9a7",  # Mad World - Gary Jules
    "bc10b379-413a-49d9-b6a5-36771762c4d3",  # Tears in Heaven - Eric Clapton
]
sad_mbids = [
    "25d8de5e-3662-4ffd-8dea-511a696ac3e7",  # Hurt - Johnny Cash
    "b9109195-0449-4900-9399-339882b38c29",  # Everybody Hurts - R.E.M.
    "028efe7f-cdfb-4135-846f-848f2fff15b1",  # Someone Like You - Adele
    "31fb9fce-395c-4556-83c7-f0a2abf12b3a",  # Nothing Compares 2 U - Sinéad O'Connor
    "bc10b379-413a-49d9-b6a5-36771762c4d3",  # Tears in Heaven - Eric Clapton
    "49eb0679-0a3d-4be5-9adf-973cd429b9a7",  # Mad World - Gary Jules
    "a478f801-521b-40e7-8bb4-5faeffb1f3e4",  # The Scientist - Coldplay
    "760828c4-c062-4667-94c6-0d358c1c768c",  # River - Joni Mitchell
    "f367ad6e-efd5-4496-aba5-21a4edcd82c6",  # Happy - Pharrell Williams
    "d4009c09-a339-4b89-bc45-d91ab649acb6",  # Don't Stop Me Now - Queen
    "3ad31947-ae54-4409-aeaa-8dbd16c3825f",  # Good Vibrations - The Beach Boys
]
relaxed_mbids = [
    "8121ce09-135d-477d-aafb-0404f1f15b78",  # Blue in Green - Miles Davis
    "306442b9-13d3-4e7c-b2c9-dcbe12ad0b22",  # An Ending (Ascent) - Brian Eno
    "6e686d11-0191-4021-9bb7-a46535709ca4",  # Watermark - Enya
    "b79e4c16-c6ed-49b8-b867-1a89ebe6ddd1",  # Clair De Lune - Claude Debussy
    "a38c28ee-4430-4429-b3e7-e7920895b2d2",  # Come Away With Me - Norah Jones
    "548bfe47-8d1c-4a4d-8a0f-2f9a4ee6af5b",  # Better Together - Jack Johnson
    "864c7424-c9ab-481e-a9c0-ad986e646c67",  # La Femme d'argent - Air
    "120bdf8e-c760-4aaf-8e34-ddd1ab3be504",  # Territorial Pissings - Nirvana
    "3dabfc40-33b4-4dba-8d94-79e77c4d1448",  # Firestarter - The Prodigy
    "1675aafe-397d-41fc-b7f6-fde5681f56c4",  # Raining Blood - Slayer
]
party_mbids = [
    "7c5b6609-0e04-4ea4-a871-513301db8c9a",  # Party Rock Anthem - LMFAO
    "7d9d4262-656c-4e93-ad42-0e0ccfbf0635",  # Celebration - Kool & The Gang
    "4f595ab5-18bb-4007-b17d-066a0e704234",  # Yeah! - Usher
    "4c0566be-d3d9-42b2-964d-cf349972123b",  # Don't Stop the Music - Rihanna
    "ef96f0f7-b26b-421e-a0e3-b9fdb45cd500",  # Get the Party Started - P!nk
    "0810268f-2e52-4f89-86be-42762ea3f6d4",  # Just Dance - Lady Gaga
    "abcb30ab-58a7-4f63-9699-952b9b422fc2",  # I Gotta Feeling - The Black Eyed Peas
    "b75c56c6-cbfc-4d1a-83fd-b2e3f97194cf",  # On the Nature of Daylight - Max Richter
    "dbdd9b68-8971-4ff8-abec-40a00119dd82",  # Strange Fruit - Billie Holiday
    "9c9ee25e-f321-4d35-b126-2316c488beae",  # Gymnopédie no. 1 - Erik Satie
]
acoustic_examples = {
    "b4184747-0157-41be-ae44-e3a6a0c13272": "Studio",  # Come as You Are - Nirvana
    "9738401c-5720-4569-8f56-9780b8b0ab2e": "Unplugged",  # Come as You Are - Nirvana
    "70595637-9310-45f2-a266-58f8de4874a7": "Studio",  # Creep - Radiohead
    "7cdf2e8e-f52c-4008-a493-d37b41fc1004": "Acoustic version",  # Creep (acoustic) - Radiohead
    "8121ce09-135d-477d-aafb-0404f1f15b78": "Acoustic",  # Blue in Green - Miles Davis
    "dbdd9b68-8971-4ff8-abec-40a00119dd82": "Acoustic",  # Strange Fruit - Billie Holiday
    "9c9ee25e-f321-4d35-b126-2316c488beae": "Acoustic",  # Gymnopédie no. 1 - Erik Satie
    "17416564-fb6e-471c-a4ad-0bfc45292cd9": "Acoustic", # Blackbird - Beatles
    "60fa767a-d85d-4991-82bc-4294e0b11ae7": "Electronic",  # One More Time - Daft Punk
    "3dabfc40-33b4-4dba-8d94-79e77c4d1448": "Electronic",  # Firestarter - The Prodigy
    "864c7424-c9ab-481e-a9c0-ad986e646c67": "Hybrid",  # La Femme d’argent - Air
    "a7ed3365-affa-4331-8bd9-1ececaf9ae77": "Hybrid",  # The Rip - Portishead
}
acoustic_mbids = list(acoustic_examples)
electronic_examples = {
    "65bf94b8-6d5d-4254-aa5f-a939b1794303": "Electronic",  # Hyperballad - Björk
    "896b2ffc-5358-4942-b2df-b37d7148ebef": "String quartet version",  # Hyperballad - Björk
    "28b9dce4-05f9-4fe0-8eb1-55c824310f1b": "Electronic",  # Take On Me - a-ha
    "e3982390-ef01-4ebb-a84b-2a1fdbbd9f96": "Unplugged",  # Take On Me - a-ha
    "c3425799-284e-454a-a006-873c953f1cad": "Electronic",  # The Robots - Kraftwerk
    "60fa767a-d85d-4991-82bc-4294e0b11ae7": "Electronic",  # One More Time - Daft Punk
    "550a5e60-445c-446e-a3d0-390c1ca5dfc7": "Electronic", # Popcorn - Hot Butter
    "9c9ee25e-f321-4d35-b126-2316c488beae": "Acoustic",  # Gymnopédie no. 1 - Erik Satie
    "b4184747-0157-41be-ae44-e3a6a0c13272": "Amplified rock",  # Come as You Are - Nirvana
    "864c7424-c9ab-481e-a9c0-ad986e646c67": "Hybrid",  # La Femme d’argent - Air
    "a7ed3365-affa-4331-8bd9-1ececaf9ae77": "Hybrid",  # The Rip - Portishead
}
electronic_mbids = list(electronic_examples)
instrumental_examples = {
    "3dabfc40-33b4-4dba-8d94-79e77c4d1448": "Vocal version",  # Firestarter - The Prodigy
    "82e46ba4-b677-480b-baca-7fe8270e34bf": "Instrumental version",  # Firestarter - The Prodigy
    "8121ce09-135d-477d-aafb-0404f1f15b78": "Instrumental",  # Blue in Green - Miles Davis
    "9c9ee25e-f321-4d35-b126-2316c488beae": "Instrumental",  # Gymnopédie no. 1 - Erik Satie
    "028efe7f-cdfb-4135-846f-848f2fff15b1": "Sung vocals",  # Someone Like You - Adele
    "dbdd9b68-8971-4ff8-abec-40a00119dd82": "Sung vocals",  # Strange Fruit - Billie Holiday
    "73b01cea-2dad-4fc2-9e61-02a31477c1b1": "Wordless vocals",  # The Great Gig in the Sky - Pink Floyd
    "6a89527f-8994-412b-ac5b-36e098d0bef8": "Wordless vocals",  # Boadicea - Enya
    "ea8095ef-700b-4a06-9f40-9b22120c3d32": "Spoken samples",  # Frontier Psychiatrist - The Avalanches
    "f1a6a40f-78f5-4918-968d-f64363bae94c": "Processed vocals",  # Harder, Better, Faster, Stronger - Daft Punk
}
instrumental_mbids = list(instrumental_examples)
# Deduplicate mbids while preserving order for consistency between runs
target_mbids = list(dict.fromkeys(
    danceable_mbids + aggressive_mbids + happy_mbids + sad_mbids + relaxed_mbids
    + party_mbids + acoustic_mbids + electronic_mbids + instrumental_mbids
))

feature_stats = analysis_helpers.aggregate_stats(target_mbids, use_sample=False, num_parts=None)

In [ ]:
# Show names of features and the shapes of the raw and normalized feature matrices.
features_raw = feature_store["feature_matrix_raw"]
features_norm = feature_store["feature_matrix"]
feature_names = feature_store["feature_names"]

display(Markdown("**Feature matrix overview**"))
print("Audio features: \n", feature_names)
print("Shape of raw feature matrix: ", features_raw.shape)
print("Shape of normalized feature matrix:", features_norm.shape)

In [ ]:
# Generate standardized matrix from raw features
scaler = StandardScaler(with_mean=True, with_std=True).fit(features_raw)
features_scaled = scaler.transform(features_raw).astype(np.float32)

# Re-generate the normalized matrix so we can compare against the one saved in the store
temp_features_norm = features_scaled / (
    np.linalg.norm(features_scaled, axis=1, keepdims=True) + 1e-8
)

# Check normalized matrix in store is correct
assert np.array_equal(temp_features_norm, features_norm) is True

feature_df = pd.DataFrame(features_raw, columns=feature_names)

display(Markdown("**Values in the raw feature matrix**"))
display(feature_df.head())

In [ ]:
basic_stats = feature_df.agg(["mean", "std", "min", "max",]).T

with pd.option_context('display.float_format', lambda x: f'{x:,.4f}'):
    display(Markdown("**Basic stats for raw feature matrix**"))
    display(basic_stats)

In [ ]:
percentiles = (
    feature_df.quantile([0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]).T
    .rename(columns=lambda x: f"{x:.0%}")
)

display(Markdown(
    "**Percentiles for the raw feature matrix**\n\n"
    "Each column gives the score at that percentile across tracks. "
    "Values are rounded to four decimal places, so `0.0000` and `1.0000` can represent scores close to the endpoints."
))
with pd.option_context('display.float_format', lambda x: f'{x:,.4f}'):
    display(percentiles)

#### Danceability

In [ ]:
# Compare danceability histogram between raw, scaled and normalized matrices
danc_raw = features_raw[:, feature_names == "danceability"].flatten()
danc_scaled = features_scaled[:, feature_names == "danceability"].flatten()
danc_normalized = features_norm[:, feature_names == "danceability"].flatten()

fig, axes = plt.subplots(1, 2, figsize=(14, 7))

axes[0].hist(
    danc_raw, bins=200, density=True, histtype="step", color="tab:blue", label="Raw"
)
axes[0].hist(
    danc_scaled, bins=200, density=True, histtype="step", color="tab:orange", label="Scaled",
)
axes[0].set_title("Danceability Raw vs Scaled")
axes[0].legend()

axes[1].hist(
    danc_raw, bins=200, density=True, histtype="step", color="tab:blue", label="Raw"
)
axes[1].hist(
    danc_normalized, bins=200, density=True, histtype="step", color="tab:olive", label="Normalized",
)
axes[1].set_title("Danceability Raw vs Normalized")
axes[1].legend()

for axis in axes:
    axis.set_xlabel("Danceability value")
    axis.set_ylabel("Density")
plt.show()

In [ ]:
danc_stats_df = pd.DataFrame({
    "min": [danc_raw.min(), danc_scaled.min(), danc_normalized.min()],
    "max": [danc_raw.max(), danc_scaled.max(), danc_normalized.max()],
    "mean": [danc_raw.mean(), danc_scaled.mean(), danc_normalized.mean()],
    "std": [danc_raw.std(), danc_scaled.std(), danc_normalized.std()]
}, index=["raw", "scaled", "normalized"])

with pd.option_context('display.float_format', lambda x: f'{x:,.4f}'):
    display(Markdown("**Basic stats for danceability**"))
    display(danc_stats_df)

In [ ]:
near_zero = (danc_raw < 0.01).mean() * 100
nearish_zero = (danc_raw < 0.1).mean() * 100
near_one = (danc_raw > 0.99).mean() * 100
nearish_one = (danc_raw > 0.9).mean() * 100
uncertain = ((danc_raw > 0.4) & (danc_raw < 0.6)).mean() * 100

print(f"{fg.WHITE}Below 0.01: {fg.GREEN}{near_zero:.2f}%{fg.RESET}")
print(f"{fg.WHITE}Below 0.10: {fg.GREEN}{nearish_zero:.2f}%{fg.RESET}")
print(f"{fg.WHITE}Above 0.99: {fg.GREEN}{near_one:.2f}%{fg.RESET}")
print(f"{fg.WHITE}Above 0.90: {fg.GREEN}{nearish_one:.2f}%{fg.RESET}")
print(f"Uncertain predictions: {fg.MAGENTA}{uncertain:.2f}%{fg.RESET}")

In [ ]:
# Check the danceability of known tracks
danceable_data = []

for mbid in danceable_mbids:
    bytes = np.frombuffer(uuid.UUID(mbid).bytes, dtype="V16", count=1)[0]
    index = np.where(feature_store["mbids"] == bytes)[0][0]

    track = Track.objects.filter(musicbrainz_recordingid=mbid).prefetch_related("artists").first()
    if not track:
        continue

    artists = ", ".join(artist.name for artist in track.artists.all())

    danceable_data.append({
        "title": track.title, "artists": artists,
        "raw": danc_raw[index], "scaled": danc_scaled[index], "norm": danc_normalized[index],
        # "mbid": mbid
    })

danc_df = pd.DataFrame(danceable_data)

print(f"{bg.BLACK}{style.BRIGHT}{fg.GREEN}The merged danceability values for known tracks \
- based on feature matrix contents{style.RESET_ALL}")
with pd.option_context('display.float_format', lambda x: f'{x:,.5f}'):
    display(danc_df)

**How to read the submission tables**

The feature matrix stores one merged score per feature for each recording. The tables below summarize the individual AcousticBrainz classifier predictions for that recording.

`total_pred` counts all valid predictions for the feature, including ties. A prediction counts toward a category only when that category has the higher score. `NaN` in a summary statistic means the group had fewer than two predictions.

In [ ]:
df = export_stats(feature_stats, feature="danceability", target_mbids=danceable_mbids)

print(f"{bg.BLACK}{style.BRIGHT}{fg.GREEN}Danceability stats based on duplicate submissions found \
in the original dataset{style.RESET_ALL}")

print(f"{style.BRIGHT}Predictions{style.RESET_ALL}")
display(df[["title", "artists", "total_pred", "danceable_count", "not_danceable_count"]])

print(f"{style.BRIGHT}Statistics for submissions tagged {fg.GREEN}danceable{style.RESET_ALL}")
display(df[[
    "title", "artists", "tagged_danceable_mean", "tagged_danceable_median", "tagged_danceable_std"
]])

print(f"{style.BRIGHT}Statistics for submissions tagged {fg.GREEN}not_danceable{style.RESET_ALL}")
display(df[[
    "title", "artists", "tagged_not_danceable_mean", "tagged_not_danceable_median",
    "tagged_not_danceable_std"
]])

print(f"{style.BRIGHT}Statistics for all submissions (regardless of tag)")
display(df[[
    "title", "artists", "all_danceable_mean", "all_danceable_median", "all_danceable_std",
    "all_not_danceable_mean", "all_not_danceable_median", "all_not_danceable_std"
]])

##### Conclusions
Values for the danceability feature are heavily concentrated at the extremes:
 - $~30\%$ of them being $<0.01$ or $>0.99$
 - $~60\%$ of them being $<0.10$ or $>0.90$
 
Clearly they were generated by a classifier that was determining if a song was danceable or not. The feature doesn't show _how_ danceable the song is.

This brings into question how we should view the features. So far we've assumed they were continuous variables that mapped to some sort of distribution (normal distribution). But the findings show that they're clustered at the extremes.

#### Aggressiveness

In [ ]:
# Compare aggressiveness histogram between raw, scaled and normalized matrices
aggr_raw = features_raw[:, feature_names == "aggressiveness"].ravel()
aggr_scaled = features_scaled[:, feature_names == "aggressiveness"].ravel()
aggr_normalized = features_norm[:, feature_names == "aggressiveness"].ravel()

fig, axes = plt.subplots(1, 2, figsize=(14, 7))

axes[0].hist(
    aggr_raw, bins=200, density=True, histtype="step", color="tab:blue", label="Raw",
)
axes[0].hist(
    aggr_scaled, bins=200, density=True, histtype="step", color="tab:orange", label="Scaled",
)
axes[0].set_title("Aggressiveness Raw vs Scaled")
axes[0].legend()

axes[1].hist(
    aggr_raw, bins=200, density=True, histtype="step", color="tab:blue", label="Raw",
)
axes[1].hist(
    aggr_normalized, bins=200, density=True, histtype="step", color="tab:olive", label="Normalized",
)
axes[1].set_title("Aggressiveness Raw vs Normalized")
axes[1].legend()

for axis in axes:
    axis.set_xlabel("Aggressiveness value")
    axis.set_ylabel("Density")
plt.show()

In [ ]:
aggr_stats_df = pd.DataFrame({
    "min": [aggr_raw.min(), aggr_scaled.min(), aggr_normalized.min()],
    "max": [aggr_raw.max(), aggr_scaled.max(), aggr_normalized.max()],
    "mean": [aggr_raw.mean(), aggr_scaled.mean(), aggr_normalized.mean()],
    "std": [aggr_raw.std(), aggr_scaled.std(), aggr_normalized.std()]
}, index=["raw", "scaled", "normalized"])

with pd.option_context('display.float_format', lambda x: f'{x:,.7f}'):
    display(Markdown("**Basic stats for aggressiveness**"))
    display(aggr_stats_df)

In [ ]:
near_zero = (aggr_raw < 0.01).mean() * 100
nearish_zero = (aggr_raw < 0.1).mean() * 100
near_one = (aggr_raw > 0.99).mean() * 100
nearish_one = (aggr_raw > 0.9).mean() * 100
uncertain = ((aggr_raw > 0.4) & (aggr_raw < 0.6)).mean() * 100

print(f"{fg.WHITE}Below 0.01: {fg.GREEN}{near_zero:.2f}%{fg.RESET}")
print(f"{fg.WHITE}Below 0.10: {fg.GREEN}{nearish_zero:.2f}%{fg.RESET}")
print(f"{fg.WHITE}Above 0.99: {fg.GREEN}{near_one:.2f}%{fg.RESET}")
print(f"{fg.WHITE}Above 0.90: {fg.GREEN}{nearish_one:.2f}%{fg.RESET}")
print(f"Between 0.4 and 0.6: {fg.MAGENTA}{uncertain:.2f}%{fg.RESET}")

In [ ]:
# Check the aggressiveness of known tracks
aggressive_data = []

for mbid in aggressive_mbids:
    bytes = np.frombuffer(uuid.UUID(mbid).bytes, dtype="V16", count=1)[0]
    index = np.where(feature_store["mbids"] == bytes)[0][0]

    track = Track.objects.filter(musicbrainz_recordingid=mbid).prefetch_related("artists").first()
    if not track:
        continue

    artists = ", ".join(artist.name for artist in track.artists.all())

    aggressive_data.append({
        "title": track.title, "artists": artists,
        "raw": aggr_raw[index], "scaled": aggr_scaled[index], "norm": aggr_normalized[index],
        # "mbid": mbid
    })

aggr_df = pd.DataFrame(aggressive_data)

print(f"{bg.BLACK}{style.BRIGHT}{fg.GREEN}The merged aggressiveness values for known tracks \
- based on feature matrix contents{style.RESET_ALL}")
with pd.option_context('display.float_format', lambda x: f'{x:,.5f}'):
    display(aggr_df)

In [ ]:
df = export_stats(feature_stats, feature="mood_aggressive", target_mbids=aggressive_mbids)

print(f"{bg.BLACK}{style.BRIGHT}{fg.GREEN}Aggressiveness stats based on duplicate submissions found \
in the original dataset{style.RESET_ALL}")

print(f"{style.BRIGHT}Predictions{style.RESET_ALL}")
display(df[["title", "artists", "total_pred", "aggressive_count", "not_aggressive_count"]])

print(f"{style.BRIGHT}Statistics for submissions tagged {fg.GREEN}aggressive{style.RESET_ALL}")
display(df[[
    "title", "artists", "tagged_aggressive_mean", "tagged_aggressive_median", "tagged_aggressive_std"
]])

print(f"{style.BRIGHT}Statistics for submissions tagged {fg.GREEN}not_aggressive{style.RESET_ALL}")
display(df[[
    "title", "artists", "tagged_not_aggressive_mean", "tagged_not_aggressive_median",
    "tagged_not_aggressive_std"
]])

print(f"{style.BRIGHT}Statistics for all submissions (regardless of tag)")
display(df[[
    "title", "artists", "all_aggressive_mean", "all_aggressive_median", "all_aggressive_std",
    "all_not_aggressive_mean", "all_not_aggressive_median", "all_not_aggressive_std"
]])

##### Conclusions

Like danceability, aggressiveness is also clustered at the extremes. We could assume that most features are like this going forward but we should verify.

Around $57\%$ of the values are less than $0.1$ indicating that most tracks don't have an aggressive mood. 
Manual inspection has shown that the feature is accurate: popular heavy-metal tracks rank high in aggressiveness while rock ballads rank low.

The low mean confirms that most values a closer to 0 than 1. Compared to the mean, the relatively large standard deviation indicates that there are large number of tracks scattered across the range, especially at the upper limit.

**Choice of method for merging duplicates**

Something warranting further investigation is "Enter Sandman" which despite being a well-known heavy metal anthem, received a low aggressiveness prediction. It might be that the slower nursery-rhyme segments confuse classifiers. Separate investigation has shown that predictions are split between aggressive / not aggressive. Each time a prediction was made, it had high average confidence for aggressive.

It's not clear that merging duplicate submissions using a median is the right choice. We should investigate if this pattern of conflicting predictions reoccurs.

#### Happiness

In [ ]:
# Compare happiness histogram between raw, scaled and normalized matrices
happy_raw = features_raw[:, feature_names == "happiness"].ravel()
happy_scaled = features_scaled[:, feature_names == "happiness"].ravel()
happy_normalized = features_norm[:, feature_names == "happiness"].ravel()

fig, axes = plt.subplots(1, 2, figsize=(14, 7))

axes[0].hist(
    happy_raw, bins=200, density=True, histtype="step", color="tab:blue", label="Raw",
)
axes[0].hist(
    happy_scaled, bins=200, density=True, histtype="step", color="tab:orange", label="Scaled",
)
axes[0].set_title("Happiness Raw vs Scaled")
axes[0].legend()

axes[1].hist(
    happy_raw, bins=200, density=True, histtype="step", color="tab:blue", label="Raw",
)
axes[1].hist(
    happy_normalized, bins=200, density=True, histtype="step", color="tab:olive", label="Normalized",
)
axes[1].set_title("Happiness Raw vs Normalized")
axes[1].legend()

for axis in axes:
    axis.set_xlabel("Happiness value")
    axis.set_ylabel("Density")
plt.show()

In [ ]:
happy_stats_df = pd.DataFrame({
    "min": [happy_raw.min(), happy_scaled.min(), happy_normalized.min()],
    "max": [happy_raw.max(), happy_scaled.max(), happy_normalized.max()],
    "mean": [happy_raw.mean(), happy_scaled.mean(), happy_normalized.mean()],
    "std": [happy_raw.std(), happy_scaled.std(), happy_normalized.std()]
}, index=["raw", "scaled", "normalized"])

with pd.option_context('display.float_format', lambda x: f'{x:,.7f}'):
    display(Markdown("**Basic stats for happiness**"))
    display(happy_stats_df)

In [ ]:
near_zero = (happy_raw < 0.01).mean() * 100
nearish_zero = (happy_raw < 0.1).mean() * 100
near_one = (happy_raw > 0.99).mean() * 100
nearish_one = (happy_raw > 0.9).mean() * 100
uncertain = ((happy_raw > 0.4) & (happy_raw < 0.6)).mean() * 100

print(f"{fg.WHITE}Below 0.01: {fg.GREEN}{near_zero:.2f}%{fg.RESET}")
print(f"{fg.WHITE}Below 0.10: {fg.GREEN}{nearish_zero:.2f}%{fg.RESET}")
print(f"{fg.WHITE}Above 0.99: {fg.GREEN}{near_one:.2f}%{fg.RESET}")
print(f"{fg.WHITE}Above 0.90: {fg.GREEN}{nearish_one:.2f}%{fg.RESET}")
print(f"Between 0.4 and 0.6: {fg.MAGENTA}{uncertain:.2f}%{fg.RESET}")

In [ ]:
# Check the happiness of known tracks
happy_data = []

for mbid in happy_mbids:
    mbid_bytes = np.frombuffer(uuid.UUID(mbid).bytes, dtype="V16", count=1)[0]
    index = np.where(feature_store["mbids"] == mbid_bytes)[0][0]

    track = Track.objects.filter(musicbrainz_recordingid=mbid).prefetch_related("artists").first()
    if not track:
        continue

    artists = ", ".join(artist.name for artist in track.artists.all())

    happy_data.append({
        "title": track.title, "artists": artists, "submissions": track.submissions,
        "raw": happy_raw[index], "scaled": happy_scaled[index], "norm": happy_normalized[index],
        "mbid": mbid
    })

happy_df = pd.DataFrame(happy_data)

print(f"{bg.BLACK}{style.BRIGHT}{fg.GREEN}The merged happiness values for known tracks \
- based on feature matrix contents{style.RESET_ALL}")
with pd.option_context('display.float_format', lambda x: f'{x:,.5f}'):
    display(happy_df)

In [ ]:
df = export_stats(feature_stats, feature="mood_happy", target_mbids=happy_mbids)

print(f"{bg.BLACK}{style.BRIGHT}{fg.GREEN}Happiness stats based on duplicate submissions found \
in the original dataset{style.RESET_ALL}")

print(f"{style.BRIGHT}Predictions{style.RESET_ALL}")
display(df[["title", "artists", "total_pred", "happy_count", "not_happy_count"]])

print(f"{style.BRIGHT}Statistics for submissions tagged {fg.GREEN}happy{style.RESET_ALL}")
display(df[[
    "title", "artists", "tagged_happy_mean", "tagged_happy_median", "tagged_happy_std"
]])

print(f"{style.BRIGHT}Statistics for submissions tagged {fg.GREEN}not_happy{style.RESET_ALL}")
display(df[[
    "title", "artists", "tagged_not_happy_mean", "tagged_not_happy_median",
    "tagged_not_happy_std"
]])

print(f"{style.BRIGHT}Statistics for all submissions (regardless of tag)")
display(df[[
    "title", "artists", "all_happy_mean", "all_happy_median", "all_happy_std",
    "all_not_happy_mean", "all_not_happy_median", "all_not_happy_std"
]])

##### Conclusions

Happiness is much less concentrated at the extremes than the previous features: only $6.06\%$ of raw values are below $0.01$ or above $0.99$, while $71.74\%$ lie between $0.1$ and $0.9$. The earlier assumption that most features would be clustered at the endpoints does not hold here.

The known-track check gives some counterintuitive scores: 
- *Pharrell Williams' "Happy"* is $0.378$ and *The Beach Boys' "Good Vibrations"* is $0.184$, while Gary Jules' "Mad World" is $0.507$. 
- For *"Happy"*, the source predictions also lean strongly toward `not_happy`: $289$ votes versus $35$ happy votes, with 2 ties. These examples warrant a listening check before relying on the feature to rank tracks by perceived happiness.

*TL;DR Good spread, mid usefulness.*

#### Sadness
Note: It represents the fact the the song has a sad mood or not. It's not the mathematical inverse of the previous feature.

In [ ]:
# Compare sadness histogram between raw, scaled and normalized matrices
sad_raw = features_raw[:, feature_names == "sadness"].ravel()
sad_scaled = features_scaled[:, feature_names == "sadness"].ravel()
sad_normalized = features_norm[:, feature_names == "sadness"].ravel()

fig, axes = plt.subplots(1, 2, figsize=(14, 7))

axes[0].hist(
    sad_raw, bins=200, density=True, histtype="step", color="tab:blue", label="Raw",
)
axes[0].hist(
    sad_scaled, bins=200, density=True, histtype="step", color="tab:orange", label="Scaled",
)
axes[0].set_title("Sadness Raw vs Scaled")
axes[0].legend()

axes[1].hist(
    sad_raw, bins=200, density=True, histtype="step", color="tab:blue", label="Raw",
)
axes[1].hist(
    sad_normalized, bins=200, density=True, histtype="step", color="tab:olive", label="Normalized",
)
axes[1].set_title("Sadness Raw vs Normalized")
axes[1].legend()

for axis in axes:
    axis.set_xlabel("Sadness value")
    axis.set_ylabel("Density")
plt.show()

In [ ]:
sad_stats_df = pd.DataFrame({
    "min": [sad_raw.min(), sad_scaled.min(), sad_normalized.min()],
    "max": [sad_raw.max(), sad_scaled.max(), sad_normalized.max()],
    "mean": [sad_raw.mean(), sad_scaled.mean(), sad_normalized.mean()],
    "std": [sad_raw.std(), sad_scaled.std(), sad_normalized.std()]
}, index=["raw", "scaled", "normalized"])

with pd.option_context('display.float_format', lambda x: f'{x:,.7f}'):
    display(Markdown("**Basic stats for sadness**"))
    display(sad_stats_df)

In [ ]:
near_zero = (sad_raw < 0.01).mean() * 100
nearish_zero = (sad_raw < 0.1).mean() * 100
near_one = (sad_raw > 0.99).mean() * 100
nearish_one = (sad_raw > 0.9).mean() * 100
uncertain = ((sad_raw > 0.4) & (sad_raw < 0.6)).mean() * 100

print(f"{fg.WHITE}Below 0.01: {fg.GREEN}{near_zero:.2f}%{fg.RESET}")
print(f"{fg.WHITE}Below 0.10: {fg.GREEN}{nearish_zero:.2f}%{fg.RESET}")
print(f"{fg.WHITE}Above 0.99: {fg.GREEN}{near_one:.2f}%{fg.RESET}")
print(f"{fg.WHITE}Above 0.90: {fg.GREEN}{nearish_one:.2f}%{fg.RESET}")
print(f"Between 0.4 and 0.6: {fg.MAGENTA}{uncertain:.2f}%{fg.RESET}")

In [ ]:
happiness_raw = features_raw[:, feature_names == "happiness"].ravel()
sad_happy_correlation = np.corrcoef(sad_raw, happiness_raw)[0, 1]
both_high = ((sad_raw > 0.5) & (happiness_raw > 0.5)).mean() * 100
both_low = ((sad_raw < 0.5) & (happiness_raw < 0.5)).mean() * 100

print(f"Pearson correlation with happiness: {sad_happy_correlation:.3f}")
print(f"{fg.WHITE}Both probabilities above 0.5: {fg.GREEN}{both_high:.2f}%{fg.RESET}")
print(f"{fg.WHITE}Both probabilities below 0.5: {fg.GREEN}{both_low:.2f}%{fg.RESET}")

In [ ]:
# Check the sadness of known tracks
sad_data = []

for mbid in sad_mbids:
    mbid_bytes = np.frombuffer(uuid.UUID(mbid).bytes, dtype="V16", count=1)[0]
    index = np.where(feature_store["mbids"] == mbid_bytes)[0][0]

    track = Track.objects.filter(musicbrainz_recordingid=mbid).prefetch_related("artists").first()
    if not track:
        continue

    artists = ", ".join(artist.name for artist in track.artists.all())

    sad_data.append({
        "title": track.title, "artists": artists, "submissions": track.submissions,
        "raw": sad_raw[index], "happiness": happiness_raw[index],
        "scaled": sad_scaled[index], "norm": sad_normalized[index],
        # "mbid": mbid
    })

sad_df = pd.DataFrame(sad_data)

print(f"{bg.BLACK}{style.BRIGHT}{fg.GREEN}The merged sadness values for known tracks \
- based on feature matrix contents{style.RESET_ALL}")
with pd.option_context('display.float_format', lambda x: f'{x:,.5f}'):
    display(sad_df)

In [ ]:
df = export_stats(feature_stats, feature="mood_sad", target_mbids=sad_mbids)

print(f"{bg.BLACK}{style.BRIGHT}{fg.GREEN}Sadness stats based on duplicate submissions found \
in the original dataset{style.RESET_ALL}")

print(f"{style.BRIGHT}Predictions{style.RESET_ALL}")
display(df[["title", "artists", "total_pred", "sad_count", "not_sad_count"]])

print(f"{style.BRIGHT}Statistics for submissions tagged {fg.GREEN}sad{style.RESET_ALL}")
display(df[[
    "title", "artists", "tagged_sad_mean", "tagged_sad_median", "tagged_sad_std"
]])

print(f"{style.BRIGHT}Statistics for submissions tagged {fg.GREEN}not_sad{style.RESET_ALL}")
display(df[[
    "title", "artists", "tagged_not_sad_mean", "tagged_not_sad_median",
    "tagged_not_sad_std"
]])

print(f"{style.BRIGHT}Statistics for all submissions (regardless of tag)")
display(df[[
    "title", "artists", "all_sad_mean", "all_sad_median", "all_sad_std",
    "all_not_sad_mean", "all_not_sad_median", "all_not_sad_std"
]])

##### Conclusions

Sadness and happiness are distinct metrics determined in different ways. Their raw probabilities have only a weak negative correlation ($r = -0.150$). Both exceed $0.5$ for $9.13\%$ of tracks, while both fall below $0.5$ for $40.53\%$. They should be treated as separate signals when filtering or weighting recommendations.

The known-track check illustrates this overlap: *Adele's "Someone Like You"* scores $0.755$ for sadness and $0.628$ for happiness. *Johnny Cash's "Hurt"* scores $0.901$ for sadness, with all $251$ source predictions labelled `sad`. *R.E.M.'s "Everybody Hurts"* scores $0.830$, while *Gary Jules' "Mad World"* sits close to the decision boundary at $0.490$.

Overall `sadness` looks like a decent feature. It's not clustered at the extremes and gives scores that are close to expectations. The strength of those scores isn't very big but it's in the right direction.

#### Relaxedness

In [ ]:
relaxed_raw = features_raw[:, feature_names == "relaxedness"].ravel()
relaxed_scaled = features_scaled[:, feature_names == "relaxedness"].ravel()
relaxed_normalized = features_norm[:, feature_names == "relaxedness"].ravel()

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
comparisons = [
    ("Scaled", relaxed_scaled, "tab:orange"),
    ("Normalized", relaxed_normalized, "tab:olive"),
]
for axis, (label, values, color) in zip(axes, comparisons):
    axis.hist(
        relaxed_raw, bins=200, density=True, histtype="step", color="tab:blue", label="Raw"
    )
    axis.hist(values, bins=200, density=True, histtype="step", color=color, label=label)
    axis.set(title=f"Relaxedness Raw vs {label}", xlabel="Relaxedness value", ylabel="Density")
    axis.legend()
plt.show()

In [ ]:
relaxed_stats_df = pd.DataFrame(
    [
        (values.min(), values.max(), values.mean(), np.median(values), values.std())
        for values in (relaxed_raw, relaxed_scaled, relaxed_normalized)
    ],
    index=["raw", "scaled", "normalized"],
    columns=["min", "max", "mean", "median", "std"],
)

display(Markdown("**Basic stats for relaxedness**"))
with pd.option_context("display.float_format", "{:,.7f}".format):
    display(relaxed_stats_df)

In [ ]:
nearish_zero = (relaxed_raw < 0.1).mean() * 100
between_50_60 = ((relaxed_raw > 0.70) & (relaxed_raw < 0.90)).mean() * 100
above_60 = (relaxed_raw > 0.60).mean() * 100
nearish_one = (relaxed_raw > 0.9).mean() * 100
midrange = ((relaxed_raw > 0.4) & (relaxed_raw < 0.6)).mean() * 100

print(f"{fg.WHITE}Below 0.10: {fg.GREEN}{nearish_zero:.2f}%{fg.RESET}")
print(f"{fg.WHITE}Above 0.60: {fg.GREEN}{above_60:.2f}%{fg.RESET}")
print(f"{fg.WHITE}Between 0.70 and 0.90: {fg.GREEN}{between_50_60:.2f}%{fg.RESET}")
print(f"{fg.WHITE}Above 0.90: {fg.GREEN}{nearish_one:.2f}%{fg.RESET}")
print(f"Between 0.4 and 0.6: {fg.MAGENTA}{midrange:.2f}%{fg.RESET}")

In [ ]:
# Check the relaxedness of known tracks
relaxed_tracks = Track.objects.prefetch_related("artists").in_bulk(relaxed_mbids)
matrix_mbids = feature_store["mbids"]
relaxed_data = []

for mbid in relaxed_mbids:
    mbid_bytes = np.void(uuid.UUID(mbid).bytes)
    index = np.flatnonzero(matrix_mbids == mbid_bytes)[0]
    track = relaxed_tracks[mbid]
    artists = ", ".join(artist.name for artist in track.artists.all())
    relaxed_data.append({
        "title": track.title,
        "artists": artists,
        "submissions": track.submissions,
        "raw": relaxed_raw[index],
        "scaled": relaxed_scaled[index],
        "norm": relaxed_normalized[index],
    })

relaxed_df = pd.DataFrame(relaxed_data)

display(Markdown("**Merged relaxedness scores for selected tracks**"))
with pd.option_context("display.float_format", "{:,.5f}".format):
    display(relaxed_df)

In [ ]:
relaxed_submission_stats = export_stats(
    feature_stats, feature="mood_relaxed", target_mbids=relaxed_mbids
)

display(Markdown("**Relaxedness predictions from source submissions**"))
display(relaxed_submission_stats[["title", "artists", "total_pred", "relaxed_count", "not_relaxed_count"]])

display(Markdown("**Statistics for submissions tagged relaxed**"))
display(relaxed_submission_stats[[
    "title", "artists", "tagged_relaxed_mean", "tagged_relaxed_median", "tagged_relaxed_std",
]])

display(Markdown("**Statistics for submissions tagged not relaxed**"))
display(relaxed_submission_stats[[
    "title", "artists", "tagged_not_relaxed_mean", "tagged_not_relaxed_median", "tagged_not_relaxed_std",
]])

display(Markdown("**Statistics across all submissions**"))
display(relaxed_submission_stats[[
    "title", "artists", "all_relaxed_mean", "all_relaxed_median", "all_relaxed_std",
    "all_not_relaxed_mean", "all_not_relaxed_median", "all_not_relaxed_std",
]])

##### Conclusions

Relaxedness scores lean high. The raw median is $0.789$, and $26.64\%$ of recordings score above $0.90$. About $7.47\%$ fall below $0.01$ or above $0.99$, so relaxedness is less concentrated at the endpoints than danceability or aggressiveness.

The seven calm examples score between $0.952$ and $0.997$. For other tracks:
- Nirvana's "Territorial Pissings" scores $0.007$
- Slayer's "Raining Blood" scores $0.579$.

The calm examples make relaxedness promising as a soft recommendation preference, since most of the scores are likely to be high.

#### Partyness

In [ ]:
party_raw = features_raw[:, feature_names == "partyness"].ravel()
party_scaled = features_scaled[:, feature_names == "partyness"].ravel()
party_normalized = features_norm[:, feature_names == "partyness"].ravel()

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
comparisons = [
    ("Scaled", party_scaled, "tab:orange"),
    ("Normalized", party_normalized, "tab:olive"),
]
for axis, (label, values, color) in zip(axes, comparisons):
    axis.hist(
        party_raw, bins=200, density=True, histtype="step", color="tab:blue", label="Raw"
    )
    axis.hist(values, bins=200, density=True, histtype="step", color=color, label=label)
    axis.set(title=f"Partyness Raw vs {label}", xlabel="Partyness value", ylabel="Density")
    axis.legend()
plt.show()

In [ ]:
party_stats_df = pd.DataFrame(
    [
        (values.min(), values.max(), values.mean(), np.median(values), values.std())
        for values in (party_raw, party_scaled, party_normalized)
    ],
    index=["raw", "scaled", "normalized"],
    columns=["min", "max", "mean", "median", "std"],
)

display(Markdown("**Basic stats for partyness**"))
with pd.option_context("display.float_format", "{:,.7f}".format):
    display(party_stats_df)

In [ ]:
near_zero = (party_raw < 0.01).mean() * 100
nearish_zero = (party_raw < 0.1).mean() * 100
below_midpoint = (party_raw < 0.5).mean() * 100
near_one = (party_raw > 0.99).mean() * 100
nearish_one = (party_raw > 0.9).mean() * 100
midrange = ((party_raw > 0.4) & (party_raw < 0.6)).mean() * 100

print(f"{fg.WHITE}Below 0.01: {fg.GREEN}{near_zero:.2f}%{fg.RESET}")
print(f"{fg.WHITE}Below 0.10: {fg.GREEN}{nearish_zero:.2f}%{fg.RESET}")
print(f"{fg.WHITE}Below 0.50: {fg.GREEN}{below_midpoint:.2f}%{fg.RESET}")
print(f"{fg.WHITE}Above 0.99: {fg.GREEN}{near_one:.2f}%{fg.RESET}")
print(f"{fg.WHITE}Above 0.90: {fg.GREEN}{nearish_one:.2f}%{fg.RESET}")
print(f"Between 0.4 and 0.6: {fg.MAGENTA}{midrange:.2f}%{fg.RESET}")

In [ ]:
# Check the partyness of selected tracks
party_tracks = Track.objects.prefetch_related("artists").in_bulk(party_mbids)
matrix_mbids = feature_store["mbids"]
party_data = []

for mbid in party_mbids:
    mbid_bytes = np.void(uuid.UUID(mbid).bytes)
    index = np.flatnonzero(matrix_mbids == mbid_bytes)[0]
    track = party_tracks[mbid]
    artists = ", ".join(artist.name for artist in track.artists.all())
    party_data.append({
        "title": track.title,
        "artists": artists,
        "submissions": track.submissions,
        "raw": party_raw[index],
        "scaled": party_scaled[index],
        "norm": party_normalized[index],
    })

party_df = pd.DataFrame(party_data)

display(Markdown("**Merged partyness scores for selected tracks (feature matrix)**"))
with pd.option_context("display.float_format", "{:,.5f}".format):
    display(party_df)

In [ ]:
party_submission_stats = export_stats(
    feature_stats, feature="mood_party", target_mbids=party_mbids
)

display(Markdown("**Partyness predictions from source submissions**"))
display(party_submission_stats[["title", "artists", "total_pred", "party_count", "not_party_count"]])

display(Markdown("**Statistics for submissions tagged party**"))
display(party_submission_stats[[
    "title", "artists", "tagged_party_mean", "tagged_party_median", "tagged_party_std",
]])

display(Markdown("**Statistics for submissions tagged not party**"))
display(party_submission_stats[[
    "title", "artists", "tagged_not_party_mean", "tagged_not_party_median", "tagged_not_party_std",
]])

display(Markdown("**Statistics across all submissions**"))
display(party_submission_stats[[
    "title", "artists", "all_party_mean", "all_party_median", "all_party_std",
    "all_not_party_mean", "all_not_party_median", "all_not_party_std",
]])

##### Conclusions

Raw `partyness` scores lean low. The median is $0.216$, and $67.90\%$ of recordings score below $0.50$. About $39.04\%$ score below $0.10$, and $6.37\%$ score above $0.90$.

Five of the seven party examples score between $0.698$ and $0.941$, and the three contrasting tracks score below $0.05$. 
- *Lady Gaga's "Just Dance"* scores $0.470$
- *The Black Eyed Peas' "I Gotta Feeling"* scores $0.247$, with only $30$ of $442$ valid source predictions tagged party.

In this set, a cutoff of $0.50$ would miss two recognizable party tracks.

#### Acousticness

Prediction for if a recording has an acoustic mood. I omitted the positive vs negative prediction comparisons as they didn't add much value in above steps. We can include them if there's a significant number of outliers.

Added a check for how studio and acoustic versions of the same songs compare, alongside other well-known acoustic, electronic, and hybrid examples.

In [ ]:
acoustic_col = np.flatnonzero(feature_names == "acousticness")[0]
acoustic_raw = features_raw[:, acoustic_col]
acoustic_scaled = features_scaled[:, acoustic_col]
acoustic_normalized = features_norm[:, acoustic_col]

fig, axes = plt.subplots(1, 2, figsize=(14, 6), layout="constrained")
comparisons = [
    ("Standardized", acoustic_scaled, "tab:orange"),
    ("Normalized", acoustic_normalized, "tab:olive"),
]
for axis, (label, values, color) in zip(axes, comparisons):
    axis.hist(
        acoustic_raw, bins=200, density=True, histtype="step", color="tab:blue", label="Raw"
    )
    axis.hist(values, bins=200, density=True, histtype="step", color=color, label=label)
    axis.set(title=f"Acousticness Raw vs {label}", xlabel="Acousticness value", ylabel="Density")
    axis.legend()
plt.show()

In [ ]:
acoustic_stats_df = pd.DataFrame(
    [
        (values.min(), values.max(), values.mean(), np.median(values), values.std())
        for values in (acoustic_raw, acoustic_scaled, acoustic_normalized)
    ],
    index=["raw", "standardized", "normalized"],
    columns=["min", "max", "mean", "median", "std"],
)

display(Markdown("**Basic stats for acousticness**"))
with pd.option_context("display.float_format", "{:,.5f}".format):
    display(acoustic_stats_df)

acoustic_raw_shares = pd.Series({
    "Below 0.01": (acoustic_raw < 0.01).mean(),
    "Below 0.10": (acoustic_raw < 0.10).mean(),
    "At or above 0.50": (acoustic_raw >= 0.50).mean(),
    "Above 0.90": (acoustic_raw > 0.90).mean(),
    "Above 0.99": (acoustic_raw > 0.99).mean(),
}, name="Share of recordings")
display(acoustic_raw_shares.to_frame().style.format("{:.2%}"))

In [ ]:
acoustic_tracks = Track.objects.select_related("album").prefetch_related("artists").in_bulk(acoustic_mbids)
matrix_mbids = feature_store["mbids"]
acoustic_rows = []
for mbid, example in acoustic_examples.items():
    track = acoustic_tracks.get(mbid)
    mbid_bytes = np.void(uuid.UUID(mbid).bytes)
    indices = np.flatnonzero(matrix_mbids == mbid_bytes)
    if track is None or indices.size == 0:
        print(f"Recording {mbid} is missing from the database or feature matrix.")
        continue
    index = indices[0]
    acoustic_rows.append({
        "type": example,
        "title": track.title,
        "artists": ", ".join(artist.name for artist in track.artists.all()),
        "album": track.album.name if track.album else None,
        "submissions": track.submissions,
        "raw": acoustic_raw[index],
        "standardized": acoustic_scaled[index],
        "normalized": acoustic_normalized[index],
    })

acoustic_df = pd.DataFrame(acoustic_rows)
display(Markdown("**Merged acousticness scores for selected recordings**"))
with pd.option_context("display.float_format", "{:,.5f}".format):
    display(acoustic_df)

In [ ]:
# Keep the source table in the same order as the selected examples.
acoustic_stats_by_mbid = {mbid: feature_stats.get(mbid, {}) for mbid in acoustic_mbids}
acoustic_submission_stats = export_stats(acoustic_stats_by_mbid, feature="mood_acoustic")
acoustic_submission_stats.insert(0, "example", list(acoustic_examples.values()))

display(Markdown("**Acousticness predictions from source submissions**"))
display(acoustic_submission_stats[[
    "example", "title", "artists", "total_pred", "acoustic_count", "not_acoustic_count",
]])

display(Markdown("**Acoustic scores across all valid source submissions**"))
with pd.option_context("display.float_format", "{:,.5f}".format):
    display(acoustic_submission_stats[[
        "example", "title", "artists", "all_acoustic_mean", "all_acoustic_median", "all_acoustic_std",
    ]])

##### Conclusions

Raw acousticness has a median of $0.230$. About $41.33\%$ of scores fell below $0.10$ and $19.43\%$ exceeded $0.90$. The extremes are uneven, with $18.10\%$ below $0.01$ and $5.46\%$ above $0.99$. Vector normalization reduced the standard deviation from $1.000$ after standardization to $0.256$.

Both paired examples scored higher in their acoustic versions. 
- Radiohead's *Creep* went from $0.014$ to $0.996$, with $68$ of $72$ predictions tagged acoustic for the acoustic version. 
- Nirvana's *Come as You Are* went from $0.014$ to $0.465$. Of the $137$ unplugged predictions, $7$ favored acoustic and $21$ were ties. A raw cutoff of $0.50$ would exclude that unplugged performance.

The downside is that hybrid arrangements can score high too. Air's *La Femme d’argent* scored $0.885$ and Portishead's *The Rip* scored $0.931$.

#### Electronicness

In [ ]:
electronic_col = np.flatnonzero(feature_names == "electronicness")[0]
electronic_raw = features_raw[:, electronic_col]
electronic_scaled = features_scaled[:, electronic_col]
electronic_normalized = features_norm[:, electronic_col]

fig, axes = plt.subplots(1, 2, figsize=(14, 6), layout="constrained")
comparisons = [
    ("Standardized", electronic_scaled, "tab:orange"),
    ("Normalized", electronic_normalized, "tab:olive"),
]
for axis, (label, values, color) in zip(axes, comparisons):
    axis.hist(
        electronic_raw, bins=200, density=True, histtype="step", color="tab:blue", label="Raw"
    )
    axis.hist(values, bins=200, density=True, histtype="step", color=color, label=label)
    axis.set(title=f"Electronicness Raw vs {label}", xlabel="Electronicness value", ylabel="Density")
    axis.legend()
plt.show()

In [ ]:
electronic_stats_df = pd.DataFrame(
    [
        (values.min(), values.max(), values.mean(), np.median(values), values.std())
        for values in (electronic_raw, electronic_scaled, electronic_normalized)
    ],
    index=["raw", "standardized", "normalized"],
    columns=["min", "max", "mean", "median", "std"],
)

display(Markdown("**Basic stats for electronicness**"))
with pd.option_context("display.float_format", "{:,.5f}".format):
    display(electronic_stats_df)

electronic_raw_shares = pd.Series({
    "Below 0.01": (electronic_raw < 0.01).mean(),
    "Below 0.10": (electronic_raw < 0.10).mean(),
    "At or above 0.50": (electronic_raw >= 0.50).mean(),
    "Above 0.90": (electronic_raw > 0.90).mean(),
    "Above 0.99": (electronic_raw > 0.99).mean(),
}, name="Share of recordings")
display(electronic_raw_shares.to_frame().style.format("{:.2%}"))

In [ ]:
electronic_tracks = Track.objects.select_related("album").prefetch_related("artists").in_bulk(electronic_mbids)
electronic_rows = []
for mbid, example in electronic_examples.items():
    track = electronic_tracks.get(mbid)
    mbid_bytes = np.void(uuid.UUID(mbid).bytes)
    indices = np.flatnonzero(matrix_mbids == mbid_bytes)
    if track is None or indices.size == 0:
        print(f"Recording {mbid} is missing from the database or feature matrix.")
        continue
    index = indices[0]
    electronic_rows.append({
        "type": example,
        "title": track.title,
        "artists": ", ".join(artist.name for artist in track.artists.all()),
        "album": track.album.name if track.album else None,
        "submissions": track.submissions,
        "raw electronicness": electronic_raw[index],
        "standardized electronicness": electronic_scaled[index],
        "normalized electronicness": electronic_normalized[index],
    })

electronic_df = pd.DataFrame(electronic_rows)
display(Markdown("**Merged electronicness scores for selected recordings**"))
with pd.option_context("display.float_format", "{:,.5f}".format):
    display(electronic_df)

In [ ]:
# Keep the source table in the same order as the selected examples.
electronic_stats_by_mbid = {mbid: feature_stats.get(mbid, {}) for mbid in electronic_mbids}
electronic_submission_stats = export_stats(electronic_stats_by_mbid, feature="mood_electronic")
electronic_submission_stats.insert(0, "type", list(electronic_examples.values()))

display(Markdown("**Electronicness predictions from source submissions**"))
with pd.option_context("display.float_format", "{:,.5f}".format):
    display(electronic_submission_stats[[
        "type", "title", "artists", "total_pred", "electronic_count", "not_electronic_count",
        "all_electronic_median", "all_electronic_std",
    ]])

##### Conclusions

Raw electronicness had a median of $0.507$. About $50.81\%$ of recordings scored at least $0.50$, $11.87\%$ scored below $0.10$, and $20.12\%$ scored above $0.90$. Vector normalization reduced the standard deviation from $1.000$ after standardization to $0.255$.

Björk's *Hyperballad* scored $0.903$ in the original and $0.029$ in the Brodsky Quartet version. All $201$ valid source predictions for the original favored electronic, compared with $3$ of $35$ for the quartet version. The studio version of a-ha's *Take On Me* scored $0.188$ and only $12$ of $384$ source predictions favored electronic. A raw cutoff of $0.50$ would exclude that synth-pop recording.

#### Instrumentalness

In [ ]:
instrumental_col = np.flatnonzero(feature_names == "instrumentalness")[0]
instrumental_raw = features_raw[:, instrumental_col]
instrumental_scaled = features_scaled[:, instrumental_col]
instrumental_normalized = features_norm[:, instrumental_col]

fig, axes = plt.subplots(1, 2, figsize=(14, 6), layout="constrained")
comparisons = [
    ("Standardized", instrumental_scaled, "tab:orange"),
    ("Normalized", instrumental_normalized, "tab:olive"),
]
for axis, (label, values, color) in zip(axes, comparisons):
    axis.hist(
        instrumental_raw, bins=200, density=True, histtype="step", color="tab:blue", label="Raw"
    )
    axis.hist(values, bins=200, density=True, histtype="step", color=color, label=label)
    axis.set(title=f"Instrumentalness Raw vs {label}", xlabel="Instrumentalness value", ylabel="Density")
    axis.legend()
plt.show()

In [ ]:
instrumental_stats_df = pd.DataFrame(
    [
        (values.min(), values.max(), values.mean(), np.median(values), values.std())
        for values in (instrumental_raw, instrumental_scaled, instrumental_normalized)
    ],
    index=["raw", "standardized", "normalized"],
    columns=["min", "max", "mean", "median", "std"],
)

display(Markdown("**Basic stats for instrumentalness**"))
with pd.option_context("display.float_format", "{:,.5f}".format):
    display(instrumental_stats_df)

instrumental_raw_shares = pd.Series({
    "Below 0.01": (instrumental_raw < 0.01).mean(),
    "Below 0.10": (instrumental_raw < 0.10).mean(),
    "At or above 0.50": (instrumental_raw >= 0.50).mean(),
    "Above 0.90": (instrumental_raw > 0.90).mean(),
    "Above 0.99": (instrumental_raw > 0.99).mean(),
}, name="Share of recordings")
display(instrumental_raw_shares.to_frame().style.format("{:.2%}"))

In [ ]:
instrumental_tracks = Track.objects.select_related("album").prefetch_related("artists").in_bulk(instrumental_mbids)
instrumental_rows = []
for mbid, example in instrumental_examples.items():
    track = instrumental_tracks.get(mbid)
    mbid_bytes = np.void(uuid.UUID(mbid).bytes)
    indices = np.flatnonzero(matrix_mbids == mbid_bytes)
    if track is None or indices.size == 0:
        print(f"Recording {mbid} is missing from the database or feature matrix.")
        continue
    index = indices[0]
    instrumental_rows.append({
        "type": example,
        "title": track.title,
        "artists": ", ".join(artist.name for artist in track.artists.all()),
        "album": track.album.name if track.album else None,
        "submissions": track.submissions,
        "raw": instrumental_raw[index],
        "standardized": instrumental_scaled[index],
        "normalized": instrumental_normalized[index],
    })

instrumental_df = pd.DataFrame(instrumental_rows)
display(Markdown("**Merged instrumentalness scores for selected recordings**"))
with pd.option_context("display.float_format", "{:,.5f}".format):
    display(instrumental_df)

In [ ]:
# Keep the source table in the same order as the selected examples.
instrumental_stats_by_mbid = {mbid: feature_stats.get(mbid, {}) for mbid in instrumental_mbids}
instrumental_submission_stats = export_stats(instrumental_stats_by_mbid, feature="voice_instrumental")
instrumental_submission_stats.insert(0, "type", list(instrumental_examples.values()))

display(Markdown("**Instrumentalness predictions from source submissions**"))
with pd.option_context("display.float_format", "{:,.5f}".format):
    display(instrumental_submission_stats[[
        "type", "title", "artists", "total_pred", "instrumental_count", "voice_count",
        "all_instrumental_median", "all_instrumental_std",
    ]])

##### Conclusions

Raw instrumentalness scores clustered at both ends. The median was $0.477$, with $29.52\%$ of recordings below $0.10$ and $30.15\%$ above $0.90$. Vector normalization reduced the standard deviation from $1.000$ after standardization to $0.264$.

The instrumental version of The Prodigy's *Firestarter* scored $0.721$, up from $0.379$ for the vocal version. All valid submissions for Miles Davis's *Blue in Green* and Erik Satie's *Gymnopédie no. 1* favored instrumental, while $15$ of $437$ did so for Adele's *Someone Like You*.

Recordings with audible singing can score high. Billie Holiday's *Strange Fruit* scored $0.896$ and Daft Punk's *Harder, Better, Faster, Stronger* scored $0.982$. For two wordless performances, $193$ of $315$ predictions favored voice on Pink Floyd's *The Great Gig in the Sky*, and all $178$ favored instrumental on Enya's *Boadicea*.

### Missing Data

#### Missing Release Year

In [ ]:
years: npt.NDArray[np.int16] = np.array(feature_store["years"])
num_tracks = len(years)
zero_years = np.sum(years == 0)

print(f"Tracks missing a release year: {fg.MAGENTA}{zero_years:,}{fg.RESET} / {fg.GREEN}{num_tracks:,}{fg.RESET}")

mbids_raw: npt.NDArray[np.void] = feature_store["mbids"]
zero_year_mbids_raw = random.sample(mbids_raw[years == 0].tolist(), 10)
zero_year_mbids = [str(uuid.UUID(bytes=mbid)) for mbid in zero_year_mbids_raw]

zero_year_tracks_qs = (
    Track.objects.filter(musicbrainz_recordingid__in=zero_year_mbids)
    .select_related("album")
    .prefetch_related("artists")
)

zero_year_tracks_df = pd.DataFrame(
    [
        (
            track.title,
            ", ".join(artist.name for artist in track.artists.all()),
            track.album.name if track.album else "<no album>",
            track.album.date if track.album else None,
            track.musicbrainz_recordingid,
        )
        for track in zero_year_tracks_qs
    ],
    columns=["title", "artists", "album", "album_date", "mbid"],
)

display(Markdown("**Sample of tracks missing a release year**"))
display(zero_year_tracks_df)

Due to the way we're ingesting data, there's a chance that a song might be included without a release year if it's JSON metadata didn't contain a year for the album. This is another indicator that we should only be pulling in audio features from the AB dataset and use MusicBrainz for the metadata.

In [ ]:
recording_mbids = [
    "b0054557-397c-49ff-884d-085522b851bc",
]
tracks_by_mbid = (
    Track.objects.select_related("album")
    .prefetch_related("artists")
    .in_bulk(recording_mbids)
)

recording_mbid_bytes = [uuid.UUID(mbid).bytes for mbid in recording_mbids]
matrix_mbids = feature_store["mbids"]
matrix_matches = np.isin(matrix_mbids, np.array(recording_mbid_bytes, dtype="V16"))
years_by_mbid = {
    mbid.tobytes(): int(year)
    for mbid, year in zip(matrix_mbids[matrix_matches], feature_store["years"][matrix_matches])
}

recording_rows = []
for mbid, mbid_bytes in zip(recording_mbids, recording_mbid_bytes):
    recording = tracks_by_mbid.get(mbid)
    album = recording.album if recording else None
    recording_rows.append((
        mbid,
        recording.title if recording else None,
        ", ".join(artist.name for artist in recording.artists.all()) if recording else None,
        recording.submissions if recording else None,
        album.name if album else None,
        album.date if album else None,
        years_by_mbid.get(mbid_bytes),
    ))

recording_details = pd.DataFrame(
    recording_rows,
    columns=["mbid", "title", "artists", "submissions", "album", "db_album_date", "feature_matrix_year"],
).astype({"submissions": "Int64", "feature_matrix_year": "Int64"})
display(recording_details.style.hide(axis="index").format(na_rep=""))